In [95]:
import os 
import pickle 
import time 
import streamlit as st 
from dotenv import load_dotenv 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_groq import ChatGroq

from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFaceEndpoint

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader 
from langchain_community.vectorstores import FAISS 

# This automatically loads OPENAI_API_KEY into the system environment
load_dotenv() 


True

In [96]:
# llm = ChatGroq(
#     model="groq/compound",
#     temperature=0.4, 
#     max_tokens=600
# )
llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    max_new_tokens=300,
    temperature=0.2
)

llm = ChatHuggingFace(llm=llm_endpoint)
# as prompt is gievn out of context window of llm 
# from langchain_community.document_loaders import AsyncHtmlLoader
# from langchain_community.document_transformers import Html2TextTransformer


# urls=[
#     "https://docs.langchain.com"
# ]
# loader = AsyncHtmlLoader(urls)
# raw_html = loader.load()
# transformer = Html2TextTransformer()
# data = transformer.transform_documents(raw_html)
# len(data)

loader = UnstructuredURLLoader(urls=[
    "https://www.pinecone.io/learn/vector-database/",
    "https://docs.pinecone.io/guides/get-started/concepts",
    "https://docs.pinecone.io/guides/index-data/create-an-index",
    "https://docs.pinecone.io/guides/search/semantic-search"
])

data = loader.load()



In [97]:
len(data[0].page_content)

29579

**CHUNKING**

In [98]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1600,
    chunk_overlap = 200
)

docs = text_splitter.split_documents(data)
len(docs)


56

**EMBEDDING (STORED)**

In [99]:
embed = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vec_idx_openai = FAISS.from_documents(docs,embed)

file_path = "vec_db.pkl"

with open(file_path,'wb') as f : 
    pickle.dump(vec_idx_openai,f)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [100]:
if os.path.exists(file_path):
    with open(file_path,"rb") as f:
        vec_idx = pickle.load(f)

vec_idx

**RETRIVAL**

In [107]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate



In [108]:
retriever = vec_idx.as_retriever(
    search_type = "mmr",
    search_kwargs={"k": 8 , "fetch_k" : 20}
)




**MAP PROMPT**

In [113]:
map_template = """
You are a retrieval assistant.

Question:
{question}

Chunk:
{context}

Task:
- Extract any information that could help answer the question.
- Keep relevant code snippets.
- Keep API parameters and examples.
- Return empty only if the chunk is completely unrelated.

Relevant information:
"""

map_prompt = ChatPromptTemplate.from_template(map_template)
map_chain = map_prompt | llm | StrOutputParser()

**REDUCE PROMPT**


In [114]:
reduce_template = """
You are a strict synthesis engine.

Rules:
- Use ONLY the provided filtered chunks.
- Do NOT complete partial code unless fully present in context.
- Do NOT guess missing parts.
- If the answer is incomplete, say:
  "I cannot find complete information in the provided documents."

Question:
{question}

Filtered chunks:
{context}

FINAL ANSWER (clear and complete):
"""

reduce_prompt = ChatPromptTemplate.from_template(reduce_template)
reduce_chain = reduce_prompt | llm | StrOutputParser()



In [115]:
def ask(ques:str):
    print("Retrieving relevant chunks...")
    docs = retriever.invoke(ques)
    print(f"Retrieved {len(docs)} chunks")
    
    # -------------------------------------------------------------------------------------
    # MAP step: run one LLM call per chunk
    map_inputs = [
    {
        "question" : ques,
        "context": doc.page_content[:1200]
    }
    for doc in docs
    ]
    filtered_chunks = map_chain.batch(map_inputs)
    
    # --------------------------------------------------------------------------------------
    # Clean empty outputs
    filtered_chunks = [chunk.strip() for chunk in filtered_chunks if chunk and chunk.strip()]
    if not filtered_chunks:
        return "I cannot find that in the provided documents."
        
    # --------------------------------------------------------------------------------------
    # REDUCE step: combine map outputs into one context
    
    # due to less context window of llm 
    # merged_context = "\n\n".join(
    #     f"FC{i+1}: {chunk}" for i, chunk in enumerate(filtered_chunks)
    # )
    
    merged_context = "\n\n".join(
    f"FC{i+1}: {chunk[:500]}" for i, chunk in enumerate(filtered_chunks)
)

    final_answer = reduce_chain.invoke({
        "question" : ques,
        "context" : merged_context
    })

    return final_answer
    



In [112]:

ques = "how to Search with a record ID using semantic search"
# docs = retriever.invoke(ques)
# print("Retrieved chunks:", len(docs))

ans = ask(ques)

ans


Retrieving relevant chunks...
Retrieved 8 chunks


'To search with a record ID using semantic search in Pinecone, you can follow these steps:\n\n1. **Indexing**: Ensure your data is indexed with both dense vectors (for semantic search) and string fields with `full_text_search` (for full-text search).\n\n2. **Querying**: Use the `search_records` method to perform a vector similarity search. You can also filter by metadata, such as a record ID.\n\nHere is an example of how you might structure the API call to perform a semantic search with a record ID:\n\n```python\nfrom pinecone import Pinecone\n\n# Initialize Pinecone\npc = Pinecone(api_key="YOUR_API_KEY")\n\n# Initialize the index\nindex = pc.Index(host="INDEX_HOST")\n\n# Search for the 2 records most semantically related to a query text\nresult = index.search_records(\n    namespace="__default__",\n    query={"inputs": {"text": "your query text"}, "top_k": 2}\n)\n\n# To filter by record ID, you can use the `filter` parameter\nrecord_id = "rec2"\nfiltered_result = index.search_records(